# Estacionariedad
### ADF, KPSS y transformaciones

`Fase 1 · Video 3` — serie **Forecasting de Ventas de la A a la Z**

En el **Video 2** el EDA nos mostró tendencia y estacionalidad. Aquí lo **probamos formalmente**:
qué es la estacionariedad, cómo la miden ADF y KPSS (bien especificadas), y cómo estabilizar la serie
con transformaciones y diferenciación — el requisito que ARIMA exigirá en el Video 13.

---
## 1. Librerías y carga de datos

In [ ]:
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import FuncFormatter
from statsmodels.tsa.stattools import adfuller, kpss, acf
from statsmodels.tools.sm_exceptions import InterpolationWarning
from scipy import stats
from pathlib import Path

plt.rcParams.update({
    'figure.facecolor': '#F8FAFC', 'axes.facecolor': '#F8FAFC',
    'axes.grid': True, 'grid.color': '#E2E8F0',
    'grid.linewidth': 0.8, 'font.size': 11,
})
BLUE, RED, GREEN, PURPLE, ORANGE = '#2563EB','#DC2626','#16A34A','#7C3AED','#EA580C'
money_fmt = FuncFormatter(lambda x, _: f'${x/1e6:.1f}M')
print('Librerías cargadas')

In [ ]:
def find_csv(filename='sales_history.csv', max_levels=4):
    base = Path().resolve()
    for _ in range(max_levels):
        candidate = base / 'output' / filename
        if candidate.exists():
            return candidate
        base = base.parent
    raise FileNotFoundError(f"No se encontró '{filename}'. Corre main.py primero.")

csv_path = find_csv()
df_raw   = pd.read_csv(csv_path, nrows=2)
date_col = next(c for c in df_raw.columns if 'date' in c.strip().lower())
df = pd.read_csv(csv_path, parse_dates=[date_col])
df.rename(columns={date_col: 'date'}, inplace=True)
df.sort_values('date', inplace=True)

serie = df.groupby('date')['revenue'].sum().resample('W-MON').sum()
serie.name = 'revenue'

print(f'Serie semanal: {len(serie)} observaciones | {serie.index.min().date()} → {serie.index.max().date()}')
print(f'Media: ${serie.mean():,.0f} | Mín: ${serie.min():,.0f} | Máx: ${serie.max():,.0f}')

---
## 2. Los dos tipos de no-estacionariedad

No toda no-estacionariedad es igual. Identificar el tipo determina qué transformación aplicar.

| Tipo | Síntoma | Solución |
|---|---|---|
| **No-estacionaria en media** | La media cambia con el tiempo (tendencia) | **Diferenciar** la serie |
| **No-estacionaria en varianza** | La amplitud crece con el nivel de la serie | **Transformar** (log, Box-Cox) |
| **Ambas** | Tendencia + amplitud creciente | **Transformar primero, luego diferenciar** |

Vamos a verificar qué tipo tiene nuestra serie.

---
## 3. Prueba visual — media y varianza móviles

La forma más directa de detectar no-estacionariedad:

| Comportamiento | Interpretación |
|---|---|
| Media móvil **con pendiente** | ❌ No estacionaria en media |
| Media móvil **plana** | ✅ Media constante |
| Desv. estándar móvil **creciente** | ❌ No estacionaria en varianza (heterocedasticidad) |
| Desv. estándar móvil **plana** | ✅ Varianza constante |

In [ ]:
WINDOW = 12   # 12 semanas ≈ 3 meses

roll_mean = serie.rolling(WINDOW).mean()
roll_std  = serie.rolling(WINDOW).std()

fig = plt.figure(figsize=(14, 8))
gs  = gridspec.GridSpec(2, 1, hspace=0.45)

ax1 = fig.add_subplot(gs[0])
ax1.plot(serie.index, serie.values, color=BLUE, alpha=0.5, linewidth=1, label='Serie original')
ax1.plot(roll_mean.index, roll_mean.values, color=RED, linewidth=2.5,
         label=f'Media móvil ({WINDOW} semanas)')
ax1.yaxis.set_major_formatter(money_fmt)
ax1.set_title('¿La media es constante en el tiempo?', fontsize=12, fontweight='bold')
ax1.legend(loc='upper left')

ax2 = fig.add_subplot(gs[1])
ax2.plot(roll_std.index, roll_std.values, color=GREEN, linewidth=2,
         label=f'Desv. estándar móvil ({WINDOW} semanas)')
ax2.yaxis.set_major_formatter(money_fmt)
ax2.set_title('¿La varianza es constante en el tiempo?', fontsize=12, fontweight='bold')
ax2.legend(loc='upper left')

fig.suptitle('Prueba Visual de Estacionariedad', fontsize=15, fontweight='bold')
plt.show()

# En vez de comparar la pendiente contra un número fijo en $/semana (que depende
# de la moneda y el tamaño del negocio), medimos la DERIVA TOTAL a lo largo de la
# serie como % del nivel medio. Umbral: >10% de deriva ⇒ no estacionaria.
rm = roll_mean.dropna()
rs = roll_std.dropna()
mean_slope = np.polyfit(np.arange(len(rm)), rm.values, 1)[0]
std_slope  = np.polyfit(np.arange(len(rs)), rs.values, 1)[0]

mean_drift_rel = (mean_slope * len(rm)) / serie.mean()      # deriva total / nivel medio
std_drift_rel  = (std_slope  * len(rs)) / rs.mean()          # deriva total / amplitud media
UMBRAL = 0.10

print('─' * 50)
print('  DIAGNÓSTICO VISUAL (umbrales relativos)')
print('─' * 50)
print(f'  Deriva de la media móvil : {mean_drift_rel:+.1%} del nivel medio')
print(f'  Deriva de la std móvil   : {std_drift_rel:+.1%} de la amplitud media')
print()
print(f'  No-estacionaria en media     : {"❌ SÍ" if abs(mean_drift_rel) > UMBRAL else "✅ NO"}')
print(f'  No-estacionaria en varianza  : {"❌ SÍ" if abs(std_drift_rel)  > UMBRAL else "✅ NO"}')
print('─' * 50)

---
## 4. Las pruebas formales — ADF y KPSS bien especificadas

Las pruebas estadísticas dan un veredicto objetivo. Usamos dos con hipótesis opuestas:

| Prueba | H₀ | p < 0.05 significa... |
|---|---|---|
| **ADF** | La serie **NO** es estacionaria | ✅ **SÍ** es estacionaria |
| **KPSS** | La serie **SÍ** es estacionaria | ❌ **NO** es estacionaria |

### La trampa que casi nadie explica: el parámetro `regression`

Ambas pruebas tienen un parámetro que define qué estructura determinista **se le permite** a la serie:

| `regression` | Modelo asumido | Pregunta real |
|---|---|---|
| `'c'` | Media constante | ¿Es estacionaria **en niveles**? |
| `'ct'` | Constante + tendencia lineal | ¿Es estacionaria **alrededor de una tendencia**? |

**Por qué `'ct'` puede engañarte:** si usas `'ct'`, le permites a la serie tener tendencia y luego preguntas si lo restante es estacionario. El test puede contestar "sí", pero no responde tu pregunta original.

Vamos a correr **ambas especificaciones** para ver el efecto.

In [ ]:
def run_adf(s, alpha=0.05, regression='c', label=''):
    stat, p, lags, _, crit, _ = adfuller(s.dropna(), autolag='AIC', regression=regression)
    reg_desc = {'c': 'solo constante', 'ct': 'constante + tendencia', 'n': 'sin constante'}
    print(f'  ADF | regression="{regression}" ({reg_desc.get(regression, "")}) | {label}')
    print(f'    Estadístico: {stat:.4f}  |  p-valor: {p:.4f}  |  Lags: {lags}')
    print(f'    Críticos   : { {k: round(v,3) for k,v in crit.items()} }')
    if p < alpha:
        print('    ✅ Se rechaza H₀ → ESTACIONARIA (bajo este modelo)')
    else:
        print('    ❌ No se rechaza H₀ → NO ESTACIONARIA')
    return p < alpha

def run_kpss(s, alpha=0.05, regression='c', label=''):
    reg_desc = {'c': 'solo constante', 'ct': 'constante + tendencia'}
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        stat, p, _, crit = kpss(s.dropna(), regression=regression, nlags='auto')
    interp = [w for w in caught if issubclass(w.category, InterpolationWarning)]
    print(f'  KPSS | regression="{regression}" ({reg_desc.get(regression, "")}) | {label}')
    print(f'    Estadístico: {stat:.4f}  |  p-valor: {p:.4f}')
    print(f'    Críticos   : { {k: round(v,3) for k,v in crit.items()} }')
    if interp:
        print('    ⚠️  InterpolationWarning: el p-valor es un límite, no exacto.')
    if p < alpha:
        print('    ❌ Se rechaza H₀ → NO ESTACIONARIA')
    else:
        print('    ✅ No se rechaza H₀ → ESTACIONARIA (bajo este modelo)')
    return p >= alpha

In [ ]:
print('═' * 56)
print('  RONDA 1: regression="c" (sin tendencia en el modelo)')
print('  Pregunta: ¿Es estacionaria en niveles, tal como está?')
print('═' * 56)
adf_c  = run_adf( serie, regression='c', label='serie original')
print()
kpss_c = run_kpss(serie, regression='c', label='serie original')

In [ ]:
print('═' * 56)
print('  RONDA 2: regression="ct" (con tendencia en el modelo)')
print('  Pregunta: ¿Es estacionaria alrededor de una tendencia lineal?')
print('═' * 56)
adf_ct  = run_adf( serie, regression='ct', label='serie original')
print()
kpss_ct = run_kpss(serie, regression='ct', label='serie original')

In [ ]:
print('═' * 56)
print('  INTERPRETACIÓN — Las dos rondas')
print('═' * 56)
print()
print('  regression="c"  → ¿estacionaria EN NIVELES?')
print(f'    ADF:  {"✅ sí" if adf_c  else "❌ no"}    KPSS: {"✅ sí" if kpss_c  else "❌ no"}')
print()
print('  regression="ct" → ¿estacionaria EN TENDENCIA?')
print(f'    ADF:  {"✅ sí" if adf_ct else "❌ no"}    KPSS: {"✅ sí" if kpss_ct else "❌ no"}')
print()
print('─' * 56)
print('  LECTURA:')
print('  - "c" dice no estacionaria → la verdad: hay tendencia.')
print('  - "ct" puede decir estacionaria → engaño:')
print('    le permite tener tendencia y pregunta si lo demás es ruido.')
print('─' * 56)
print()
print('  ✅ La pregunta correcta para forecasting: ¿es estacionaria')
print('     en niveles? → usa regression="c".')

---
## 5. ¿Qué hacer cuando ADF y KPSS se contradicen?

Las combinaciones posibles y qué significa cada una:

| ADF rechaza H₀? | KPSS rechaza H₀? | Diagnóstico | Acción |
|---|---|---|---|
| ❌ No | ✅ Sí | 🟢 **Estacionaria** | Modelar directo |
| ✅ Sí | ❌ No | 🔴 **No estacionaria** | Diferenciar |
| ❌ No | ❌ No | 🟡 **Diferenciación necesaria** | Diferenciar, las dos pruebas indican que falta |
| ✅ Sí | ✅ Sí | 🟡 **Estacionaria en tendencia** | Diferenciar **o** des-tendenciar manualmente |

### Función reutilizable que automatiza esta decisión

In [ ]:
def diagnostico_estacionariedad(s, alpha=0.05, regression='c', label='Serie'):
    """Combina ADF + KPSS y devuelve un diagnóstico accionable."""
    # ADF: p-valor pequeño → estacionaria
    adf_stat, adf_p, _, _, _, _ = adfuller(s.dropna(), autolag='AIC', regression=regression)
    adf_reject = adf_p < alpha
    # KPSS: p-valor pequeño → NO estacionaria
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', InterpolationWarning)
        kpss_stat, kpss_p, _, _ = kpss(s.dropna(), regression=regression, nlags='auto')
    kpss_reject = kpss_p < alpha

    if adf_reject and not kpss_reject:
        diag = 'ESTACIONARIA'
        action = 'Modelar directo'
    elif not adf_reject and kpss_reject:
        diag = 'NO ESTACIONARIA'
        action = 'Diferenciar'
    elif not adf_reject and not kpss_reject:
        diag = 'NO ESTACIONARIA (ambas pruebas indican que falta)'
        action = 'Diferenciar'
    else:
        diag = 'ESTACIONARIA EN TENDENCIA'
        action = 'Diferenciar o des-tendenciar'

    print(f'  Diagnóstico {label}: {diag}')
    print(f'    ADF p-valor : {adf_p:.4f}  → {"rechaza H₀" if adf_reject else "no rechaza"}')
    print(f'    KPSS p-valor: {kpss_p:.4f}  → {"rechaza H₀" if kpss_reject else "no rechaza"}')
    print(f'    Acción      : {action}')
    return {'estacionaria': diag.startswith('ESTACIONARIA') and 'TENDENCIA' not in diag,
            'adf_p': adf_p, 'kpss_p': kpss_p, 'accion': action}

print('─' * 56)
diagnostico_estacionariedad(serie, regression='c', label='original');
print('─' * 56)

---
## 6. Estabilizando la varianza — log y Box-Cox

Antes de diferenciar, conviene estabilizar la varianza si es heterocedástica.  
La diferenciación trabaja sobre la media, no sobre la varianza — por eso van en este orden.

### Tres opciones de transformación

| Transformación | Fórmula | Cuándo usar |
|---|---|---|
| **Logarítmica** | `log(y)` | Varianza crece linealmente con el nivel |
| **Raíz cuadrada** | `√y` | Datos de conteo o Poisson |
| **Box-Cox** | `(yᵏ - 1) / k` | General — encuentra el `k` óptimo automáticamente |

**Box-Cox** es la más poderosa porque encuentra el λ que **maximiza la normalidad** de la serie transformada.  
Casos especiales: λ=0 equivale a log, λ=0.5 equivale a raíz, λ=1 no transforma.

In [ ]:
serie_log  = np.log(serie)
serie_sqrt = np.sqrt(serie)
serie_bc, lambda_opt = stats.boxcox(serie.values)
serie_bc = pd.Series(serie_bc, index=serie.index)

print(f'Box-Cox λ óptimo encontrado: {lambda_opt:.4f}')
if abs(lambda_opt) < 0.1:
    print('  → λ ≈ 0 → equivale a transformación logarítmica')
elif abs(lambda_opt - 0.5) < 0.1:
    print('  → λ ≈ 0.5 → equivale a raíz cuadrada')
elif abs(lambda_opt - 1) < 0.1:
    print('  → λ ≈ 1 → casi no transforma (varianza ya estable)')
else:
    print(f'  → λ = {lambda_opt:.3f} → transformación personalizada óptima')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 9))
configs = [
    (axes[0,0], serie,      'Original',          BLUE),
    (axes[0,1], serie_log,  'log(y)',            PURPLE),
    (axes[1,0], serie_sqrt, '√y',                GREEN),
    (axes[1,1], serie_bc,   f'Box-Cox (λ={lambda_opt:.3f})', ORANGE),
]
for ax, s, title, color in configs:
    ax.plot(s.index, s.values, color=color, linewidth=1.2)
    rm = s.rolling(WINDOW).mean()
    rs = s.rolling(WINDOW).std()
    ax.plot(rm.index, rm.values, color=RED, linewidth=2, alpha=0.7, label='Media móvil')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper left', fontsize=9)
fig.suptitle('Comparación de Transformaciones', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Comparación numérica del CV en la primera y última ventana
def cv_ratio(s, window=WINDOW):
    rstd  = s.rolling(window).std().dropna()
    rmean = s.rolling(window).mean().dropna()
    cv = rstd / rmean.abs()
    return cv.iloc[-1] / cv.iloc[0]

print('─' * 50)
print('  Estabilidad de varianza relativa')
print('  (ratio CV final / CV inicial — más cercano a 1 = mejor)')
print('─' * 50)
for label, s in [('Original', serie), ('log(y)', serie_log),
                  ('√y', serie_sqrt), (f'Box-Cox(λ={lambda_opt:.3f})', serie_bc)]:
    print(f'  {label:<25}: {cv_ratio(s):.4f}')
print('─' * 50)

---
## 7. Estabilizando la media — diferenciación

Una vez estable la varianza, **diferenciamos** para eliminar la tendencia.

$$\Delta y_t = y_t - y_{t-1}$$

> **Qué transformación llevamos al pipeline.** En la sección 6, Box-Cox maximiza la
> normalidad, pero devuelve un λ (aquí ≈ −1.27) difícil de interpretar e invertir para
> forecasting. De aquí en adelante usamos **log** (`log1p`): es el caso particular de
> Box-Cox con λ=0, se invierte trivialmente con `expm1`, es interpretable (los cambios
> se leen ≈ en %) y es **la misma transformación que usaremos en la descomposición
> (Video 4) y en ACF/PACF (Video 5)**. Así los tres análisis trabajan sobre la misma
> serie: priorizamos consistencia e interpretabilidad sobre el último gramo de normalidad.

In [ ]:
# Estabilizamos la varianza con log (invertible con expm1; consistente con Videos 4 y 5)
serie_stab = np.log1p(serie)
log_diff1  = serie_stab.diff().dropna()
log_diff2  = serie_stab.diff().diff().dropna()

fig, axes = plt.subplots(3, 1, figsize=(14, 10))
configs = [
    (axes[0], serie_stab, 'log(1 + y) — sin diferenciar', ORANGE),
    (axes[1], log_diff1,  'Primera diferencia (d=1)',     PURPLE),
    (axes[2], log_diff2,  'Segunda diferencia (d=2)',     GREEN),
]
for ax, s, title, color in configs:
    ax.plot(s.index, s.values, color=color, linewidth=1.2)
    ax.axhline(s.mean(), color='black', linestyle='--', linewidth=1, alpha=0.6,
               label=f'Media: {s.mean():.4f}')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(loc='upper right')
fig.suptitle('Diferenciación sobre Serie log', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('─' * 56)
print('  DIAGNÓSTICO post-diferenciación')
print('─' * 56)
diagnostico_estacionariedad(serie_stab, regression='c', label='log (d=0)')
print()
diagnostico_estacionariedad(log_diff1,  regression='c', label='log + Δ1')
print()
diagnostico_estacionariedad(log_diff2,  regression='c', label='log + Δ2')
print('─' * 56)

# ADF y KPSS NO detectan raíces unitarias estacionales: la serie puede aprobar
# ambos tests y conservar intacto su ciclo anual. Lo verificamos en el lag 52.
a52  = acf(log_diff1, nlags=60, fft=True)[52]
band = 1.96 / np.sqrt(len(log_diff1))
print()
print('  OJO: ADF/KPSS son ciegos a la estacionalidad')
print('  ' + '─' * 52)
print(f'  ACF de (log + Δ1) en el rezago 52 (ciclo anual): {a52:+.3f}')
print(f'  Banda de significancia 95%: ±{band:.3f}')
if abs(a52) > band:
    print('  → El pico estacional SIGUE significativo: la serie quedó')
    print('    estacionaria EN TENDENCIA pero NO en su ciclo anual. La')
    print('    estacionalidad se trata con diferenciación estacional (D)')
    print('    o desestacionalización (Video 4), no con más diferencias d.')
else:
    print('  → Sin estacionalidad residual relevante en el lag 52.')

---
## 8. Pipeline completo de estacionarización

Encapsulamos todo el flujo en una función reutilizable.  
Pasas cualquier serie y devuelve la versión estacionarizada + metadatos del proceso.

In [ ]:
def estacionarizar(s: pd.Series, alpha: float = 0.05, max_d: int = 2, verbose: bool = True) -> dict:
    """
    Pipeline de estacionarización en media.
    1. Estabiliza la varianza con log1p (invertible con expm1)
    2. Diferencia iterativamente hasta lograr estacionariedad o llegar a max_d
       (sin sobre-diferenciar: nunca pasa de max_d)
    3. Reporta la estacionalidad residual — la diferenciación regular no la elimina
    """
    # log1p exige valores > -1; revenue es positivo. Blindaje por si acaso:
    shift = 0.0
    if (s <= -1).any():
        shift = float(-s.min() + 1)
        s = s + shift

    # Paso 1: estabilizar varianza
    s_stab = np.log1p(s)

    # Paso 2: diferenciar hasta estacionariedad, sin pasarse de max_d
    s_work = s_stab.copy()
    d = 0
    adf_p = kpss_p = float('nan')
    is_stationary = False
    for d_try in range(0, max_d + 1):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore', InterpolationWarning)
            adf_p  = adfuller(s_work.dropna(), autolag='AIC', regression='c')[1]
            kpss_p = kpss(s_work.dropna(), regression='c', nlags='auto')[1]
        is_stationary = (adf_p < alpha) and (kpss_p >= alpha)
        d = d_try
        if is_stationary:
            break
        if d_try < max_d:                 # no diferenciar más allá de max_d
            s_work = s_work.diff().dropna()

    # Paso 3: estacionalidad residual (informativa)
    seas_acf = None
    band = None
    n = len(s_work.dropna())
    if n > 60:
        seas_acf = acf(s_work.dropna(), nlags=52, fft=True)[52]
        band = 1.96 / np.sqrt(n)

    if verbose:
        print('  Pipeline de estacionarización')
        print(f'    Shift aplicado : {shift}')
        print(f'    Transformación : log1p')
        print(f'    d aplicado     : {d}')
        print(f'    ADF p-valor    : {adf_p:.4f}')
        print(f'    KPSS p-valor   : {kpss_p:.4f}')
        print(f'    Estacionaria   : {"✅ sí" if is_stationary else "❌ no — revisa la serie"}')
        if seas_acf is not None:
            flag = '⚠️ estacionalidad residual' if abs(seas_acf) > band else 'sin estacionalidad relevante'
            print(f'    ACF lag 52     : {seas_acf:+.3f} (±{band:.3f}) → {flag}')

    return {
        'serie_estacionaria': s_work,
        'transform':          'log1p',
        'd':                  d,
        'shift':              shift,
        'adf_p':              adf_p,
        'kpss_p':             kpss_p,
        'estacionaria':       is_stationary,
        'acf_lag52':          seas_acf,
    }

print('═' * 50)
print('  PIPELINE — Serie total semanal')
print('═' * 50)
result = estacionarizar(serie)
print('═' * 50)

In [ ]:
# Comparación visual: serie original vs serie estacionarizada
fig, axes = plt.subplots(2, 1, figsize=(14, 7))

axes[0].plot(serie.index, serie.values, color=BLUE, linewidth=1.2)
axes[0].plot(serie.rolling(WINDOW).mean().index,
             serie.rolling(WINDOW).mean().values, color=RED, linewidth=2, label='Media móvil')
axes[0].yaxis.set_major_formatter(money_fmt)
axes[0].set_title('Serie Original', fontsize=12, fontweight='bold')
axes[0].legend()

s_final = result['serie_estacionaria']
axes[1].plot(s_final.index, s_final.values, color=GREEN, linewidth=1.2)
axes[1].axhline(s_final.mean(), color=RED, linewidth=2, linestyle='--',
                label=f'Media: {s_final.mean():.4f}')
axes[1].set_title(
    f'Serie Estacionarizada (log1p, d={result["d"]})',
    fontsize=12, fontweight='bold')
axes[1].legend()

fig.suptitle('Antes y Después del Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Ejemplo sobre un SKU de alto volumen (demanda regular) — el caso típico de un
# tier A que merece un modelo dedicado. Lo elegimos por revenue para asegurar que
# sea una serie densa (los slow-movers intermitentes se tratan aparte, ver Video 7).
sku_target = df.groupby('sku_id')['revenue'].sum().idxmax()
serie_sku  = (
    df[df['sku_id'] == sku_target]
    .groupby('date')['revenue']
    .sum()
    .resample('W-MON')
    .sum()
)

print('═' * 50)
print(f'  PIPELINE — {sku_target} (mayor revenue)')
print('═' * 50)
result_sku = estacionarizar(serie_sku)
print('═' * 50)

---
## 9. Conclusiones y puente al Video 4

| Aspecto | Resultado |
|---|---|
| Tipo de no-estacionariedad | En media (tendencia) **y** en varianza (heterocedástica) |
| Diagnóstico visual | Media móvil con pendiente positiva, std móvil creciente |
| ADF con `regression='c'` | No rechaza H₀ → confirma no-estacionariedad |
| KPSS con `regression='c'` | Rechaza H₀ → confirma no-estacionariedad |
| Transformación de varianza | **log (`log1p`)** — invertible con `expm1` y consistente con los Videos 4 y 5 |
| Orden de diferenciación | d=1 elimina la **tendencia** |
| Estacionalidad | ⚠️ **Sigue presente tras d=1** (ACF lag 52) — ADF/KPSS no la detectan |
| Pipeline reproducible | Función `estacionarizar()` lista para cualquier serie |

### Reglas que te llevas

1. **Siempre corre ADF y KPSS juntos.** Una sola prueba puede engañarte.
2. **`regression='c'` para preguntar si es estacionaria en niveles.** `'ct'` solo si tienes razón teórica para asumir tendencia determinista.
3. **Primero estabilizas varianza (transformación), luego media (diferenciación).** No al revés.
4. **Box-Cox maximiza la normalidad, pero para el pipeline usamos log** (λ=0): es invertible, interpretable y la misma transformación de los Videos 4–5. Elige según tu objetivo: normalidad *o* interpretabilidad/consistencia.
5. **No diferencies más de lo necesario.** Sobre-diferenciar introduce ruido artificial.
6. **Estacionario ≠ sin estacionalidad.** ADF/KPSS no ven raíces unitarias *estacionales*: una serie puede pasar ambos tests y conservar su ciclo anual intacto. La estacionalidad se trata aparte — con diferenciación estacional (D) o desestacionalización (Video 4).

---

### Próximo video
**Video 4 — Descomposición de series: clásica vs. STL**
Ahora que sabemos que la serie es no-estacionaria, vamos a separarla en sus tres componentes:
tendencia, estacionalidad y residual.